# BƯỚC 5: ĐÁNH GIÁ VÀ TRỰC QUAN HÓA KẾT QUẢ TỔNG THỂ (EVALUATION)

**Mục tiêu:** Trực quan hóa hiệu suất của mô hình Học máy tốt nhất trên tập dữ liệu kiểm thử (Test set) cho **TẤT CẢ** các mã cổ phiếu. Tự động xuất 4 biểu đồ đánh giá:
1. **Confusion Matrix:** Ma trận nhầm lẫn.
2. **Trading Signals:** Điểm mua/bán AI dự báo trên nền giá thực tế.
3. **Feature Importance:** Top các đặc trưng đóng góp nhiều nhất.
4. **Learning Curve:** Đường cong học tập chống Overfitting (BẮT BUỘC dùng TimeSeriesSplit).

In [ ]:
import os
import pickle
import warnings
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import learning_curve, TimeSeriesSplit
from sklearn.metrics import accuracy_score, f1_score

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

print("--- 5.1: Đánh giá và Chọn lọc Mô hình tốt nhất ---")

target_tickers = ['VNM', 'FPT', 'VIC']
split_date = '2024-12-31'
model_names = ['logistic_regression', 'random_forest', 'xgboost']

processed_dir = '../data/processed'
model_dir = '../models'

best_models = {}
eval_data = {}

for ticker in target_tickers:
    data_path = os.path.join(processed_dir, f"{ticker}_features.csv")
    
    if not os.path.exists(data_path):
        print(f"[{ticker}] Không tìm thấy file dữ liệu, bỏ qua.")
        continue
        
    #Nạp và chia dữ liệu
    df = pd.read_csv(data_path, index_col='Date', parse_dates=True)
    X = df.drop(columns=['Target_Class'])
    y = df['Target_Class']
    
    train_mask = X.index <= split_date
    test_mask = X.index > split_date
    
    X_train, y_train = X[train_mask], y[train_mask]
    X_test, y_test = X[test_mask], y[test_mask]
    
    best_f1 = 0.0
    best_model = None
    best_model_name = ""
    
    #Duyệt qua các file model đã lưu
    for m_name in model_names:
        model_path = os.path.join(model_dir, f"{m_name}_{ticker}.pkl")
        
        if os.path.exists(model_path):
            with open(model_path, 'rb') as f:
                model = pickle.load(f)
            
            y_pred = model.predict(X_test)
            f1 = f1_score(y_test, y_pred, average='macro')
            
            if f1 > best_f1:
                best_f1 = f1
                best_model = model
                best_model_name = m_name
                
    if best_model is not None:
        best_models[ticker] = {'model': best_model, 'name': best_model_name}
        eval_data[ticker] = {
            'X_train': X_train, 'y_train': y_train, 
            'X_test': X_test, 'y_test': y_test
        }
        print(f"-> {ticker}: Chọn {best_model_name.upper()} (F1-Macro: {best_f1:.4f})")

## 5.2. Phân tích Sai số và Tín hiệu Giao dịch
- **Ma trận Nhầm lẫn (Confusion Matrix):** Xem AI nhận diện ngày thị trường giảm (0) hay ngày thị trường tăng (1) tốt hơn.
- **Tín hiệu Dự báo (Trading Signals):** Đặt các mũi tên Xanh (Đoán Tăng) và Đỏ (Đoán Giảm) đè lên đường giá chuẩn hóa để xem trực quan độ nhạy bén của AI.

In [ ]:
print("--- 5.2: Đánh giá mô hình - Confusion Matrix & Tín hiệu giao dịch ---")

output_dir = '../outputs'
os.makedirs(output_dir, exist_ok=True)

for ticker, model_info in best_models.items():
    best_model = model_info['model']
    model_name = model_info['name'].upper()
    
    #Lấy dữ liệu test
    data = eval_data[ticker]
    X_test, y_test = data['X_test'], data['y_test']
    
    #Chạy dự đoán
    y_pred = best_model.predict(X_test)
    
    #Tạo khung hình Subplots với tỷ lệ chiều rộng 1:2
    fig, axes = plt.subplots(1, 2, figsize=(18, 6), gridspec_kw={'width_ratios': [1, 2.5]})
    
    # --- Biểu đồ 1: Confusion Matrix ---
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Dự báo Giảm(0)', 'Dự báo Tăng(1)'], 
                yticklabels=['Thực tế Giảm(0)', 'Thực tế Tăng(1)'], 
                ax=axes[0], annot_kws={"size": 14})
    axes[0].set_title(f"Confusion Matrix", pad=15)
    
    # --- Biểu đồ 2: Tín hiệu giao dịch ---
    axes[1].plot(X_test.index, X_test['Close'], label='Giá đóng cửa (Scaled)', color='#7f8c8d', alpha=0.7)
    
    #Lọc và vẽ điểm Mua/Bán
    buy_signals = X_test[y_pred == 1]
    sell_signals = X_test[y_pred == 0]
    
    axes[1].scatter(buy_signals.index, buy_signals['Close'], marker='^', color='#2ecc71', s=100, label='AI báo TĂNG', zorder=5)
    axes[1].scatter(sell_signals.index, sell_signals['Close'], marker='v', color='#e74c3c', s=100, label='AI báo GIẢM', zorder=5)
    
    axes[1].set_title(f"Tín hiệu AI so với Thực tế", pad=15)
    axes[1].set_xlabel('Thời gian')
    axes[1].set_ylabel('Mức giá')
    axes[1].legend()
    axes[1].grid(True, linestyle='--', alpha=0.5)

    fig.suptitle(f"BÁO CÁO HIỆU NĂNG: {ticker} | MÔ HÌNH: {model_name}", fontsize=16, fontweight='bold')
    plt.tight_layout()
    
    #Lưu file
    save_path = os.path.join(output_dir, f"05_evaluation_dashboard_{ticker}.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"-> Đã xuất Dashboard đánh giá cho {ticker} ({model_name})")

## 5.3. Độ quan trọng Đặc trưng và Đường cong Học tập
- **Feature Importance:** Bằng chứng cho thấy quá trình tạo thêm chỉ báo (SMA, MACD, Lag) ở Bước 3 là hoàn toàn chính xác.
- **Learning Curve:** Bắt buộc sử dụng `TimeSeriesSplit(n_splits=5)` để phân tích Overfitting/Underfitting. Đây là phần "chốt hạ" chứng minh mô hình khoa học và vững chắc.

In [ ]:
print("--- Bước 15: Giải thích mô hình (Feature Importance & Learning Curve) ---")

output_dir = '../outputs'
os.makedirs(output_dir, exist_ok=True)

#Khởi tạo TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=5)

for ticker, model_info in best_models.items():
    best_model = model_info['model']
    model_name = model_info['name'].upper()
    
    data = eval_data[ticker]
    X_train, y_train = data['X_train'], data['y_train']
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # --- Biểu đồ 1: Feature Importance ---
    try:
        if hasattr(best_model, 'feature_importances_'):
            importances = best_model.feature_importances_
        else:
            importances = np.abs(best_model.coef_[0])
            
        #lấy top 10
        feat_imp = pd.Series(importances, index=X_train.columns).sort_values(ascending=False).head(10)
        
        sns.barplot(x=feat_imp.values, y=feat_imp.index, ax=axes[0], palette='viridis')
        axes[0].set_title("Top 10 Đặc trưng Quyết định", pad=15)
        axes[0].set_xlabel("Mức độ đóng góp")
        axes[0].set_ylabel("") # Bỏ nhãn trục Y vì tên cột đã tự giải thích
    except Exception as e:
        axes[0].text(0.5, 0.5, f"Không thể vẽ Feature Importance:\n{e}", ha='center', va='center')
        axes[0].set_title("Lỗi Feature Importance")

    # --- Biểu đồ 2: Learning Curve ---
    print(f"-> Đang tính toán Learning Curve cho {ticker}...")
    
    train_sizes, train_scores, test_scores = learning_curve(
        estimator=best_model,
        X=X_train, y=y_train,
        train_sizes=np.linspace(0.1, 1.0, 5),
        cv=tscv,
        scoring='f1_macro',
        n_jobs=-1
    )
    
    train_mean = np.mean(train_scores, axis=1)
    test_mean = np.mean(test_scores, axis=1)
    
    axes[1].plot(train_sizes, train_mean, 'o-', color="#e74c3c", label="Train F1-Score")
    axes[1].plot(train_sizes, test_mean, 'o-', color="#2ecc71", label="Validation F1-Score")
    
    axes[1].set_title("Đường cong Học tập (Learning Curve)", pad=15)
    axes[1].set_xlabel("Số lượng mẫu huấn luyện")
    axes[1].set_ylabel("F1-Macro Score")
    axes[1].legend()
    axes[1].grid(True, linestyle='--', alpha=0.5)
    
    fig.suptitle(f"GIẢI THÍCH MÔ HÌNH: {ticker} | {model_name}", fontsize=16, fontweight='bold')
    plt.tight_layout()
    
    #Lưu file
    save_path = os.path.join(output_dir, f"06_model_interpretation_{ticker}.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"-> Đã xuất báo cáo Giải thích mô hình cho {ticker}\n")